In [6]:
import pandas as pd
import xlrd
import datetime
import re


In [7]:
wb = xlrd.open_workbook("labor.xls")

def xldate_to_str(v, datemode):
    try:
        tup = xlrd.xldate_as_tuple(float(v), datemode)
        if tup[0]:
            return datetime.date(*tup[:3]).isoformat()
    except:
        pass
    try:
        return str(int(float(v)))
    except:
        return str(v).strip()

def is_valid_period(v):
    v = str(v).strip()
    return bool(re.match(
        r"^([A-Za-z]{3}-[A-Za-z]{3}\s+\d{4}"
        r"|[A-Za-z]{3}\s+\d{2,4}"
        r"|[A-Za-z]+\s+\d{4}$"
        r"|\d{4}-\d{2}-\d{2}"
        r"|\d{4}$"
        r"|[A-Z]\d{8,})", v))

def build_col_names(sh, header_rows, ncols):
    raw = []
    for r in header_rows:
        row_vals = [str(sh.cell_value(r, c)).strip() for c in range(ncols)]
        last = ""
        filled = []
        for v in row_vals:
            if v:
                last = v
            filled.append(last)
        raw.append(filled)
    cols = []
    for c in range(ncols):
        parts = []
        for row in raw:
            v = re.sub(r"\s+", " ", row[c]).strip() if c < len(row) else ""
            if v and v not in parts:
                parts.append(v)
        cols.append(" | ".join(parts) if parts else str(c))
    seen = {}
    out = []
    for col in cols:
        if col in seen:
            seen[col] += 1
            out.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            out.append(col)
    return out

def parse_sheet(wb, sheet_name, header_rows, data_start, period_type):
    sh = wb.sheet_by_name(sheet_name)
    cols = build_col_names(sh, header_rows, sh.ncols)
    rows = []
    for r in range(data_start, sh.nrows):
        raw = sh.cell_value(r, 0)
        if period_type == "xldate":
            period = xldate_to_str(raw, wb.datemode)
        elif period_type == "year":
            try:
                period = str(int(float(raw)))
            except:
                period = str(raw).strip()
        else:
            period = str(raw).strip()
        if not is_valid_period(period):
            continue
        rows.append([period] + [sh.cell_value(r, c) for c in range(1, sh.ncols)])
    df = pd.DataFrame(rows, columns=cols)
    df = df.rename(columns={df.columns[0]: "period"})
    df = df.loc[:, (df.columns == "period") | df.notna().any()]
    df = df.set_index("period")
    return df


In [8]:
df_lfs_summary = parse_sheet(wb, "1", [5], 9, "label")
df_lfs_by_age = parse_sheet(wb, "2", [4, 5], 9, "label")
df_ft_pt_temp = parse_sheet(wb, "3", [4, 5], 8, "label")
df_pub_priv_emp = parse_sheet(wb, "4", [4, 5], 9, "year")
df_pub_sec_industry = parse_sheet(wb, "4(1)", [4], 7, "year")
df_workforce_jobs = parse_sheet(wb, "5", [5], 8, "label")
df_jobs_by_industry = parse_sheet(wb, "6", [5], 7, "label")
df_actual_hours = parse_sheet(wb, "7", [5], 9, "label")
df_usual_hours = parse_sheet(wb, "7(1)", [4, 5, 6], 10, "label")
df_unemp_age_duration = parse_sheet(wb, "9", [4, 5, 6], 9, "label")
df_redundancies = parse_sheet(wb, "10", [4, 6], 9, "label")
df_inactivity_reasons = parse_sheet(wb, "11", [4, 5], 9, "label")
df_young_people = parse_sheet(wb, "12", [4, 5], 9, "label")
df_awe_total = parse_sheet(wb, "13", [4, 5, 6], 9, "xldate")
df_awe_bonus = parse_sheet(wb, "14", [4, 5, 6], 9, "xldate")
df_awe_regular = parse_sheet(wb, "15", [4, 5, 6], 9, "xldate")
df_awe_real = parse_sheet(wb, "16", [4, 5, 6], 9, "xldate")
df_labour_disputes = parse_sheet(wb, "18", [5], 7, "label")
df_vacancies_size = parse_sheet(wb, "19", [5], 9, "label")
df_vacancies_unemp = parse_sheet(wb, "20", [6], 8, "label")
df_vacancies_industry = parse_sheet(wb, "21", [6, 7], 11, "label")
df_regional_lfs = parse_sheet(wb, "22", [6, 7, 8], 10, "label")

df_lfs_summary.head()

,All aged 16 & over,Total economically active,Total in employment,Unemployed,Economically inactive,Economic activity,Employment,Unemployment,Economic inactivity,All aged 16 to 64,Total economically active_1,Total in employment_1,Unemployed_1,Economically inactive_1,Economic activity_1,Employment_1,Unemployment_1,Economic inactivity_1
period,,,,,,,,,,,,,,,,,,
Jan-Mar 1971,40513000.0,25593000.0,24613000.0,980000.0,14919000.0,63.2,60.8,3.8,36.8,33559000.0,25187000.0,24219000.0,967000.0,8373000.0,75.1,72.2,3.8,24.9
Feb-Apr 1971,40531000.0,25573000.0,24575000.0,998000.0,14959000.0,63.1,60.6,3.9,36.9,33565000.0,25172000.0,24187000.0,985000.0,8393000.0,75.0,72.1,3.9,25.0
Mar-May 1971,40550000.0,25582000.0,24559000.0,1023000.0,14968000.0,63.1,60.6,4.0,36.9,33570000.0,25180000.0,24171000.0,1009000.0,8390000.0,75.0,72.0,4.0,25.0
Apr-Jun 1971,40568000.0,25600000.0,24556000.0,1045000.0,14968000.0,63.1,60.5,4.1,36.9,33576000.0,25199000.0,24168000.0,1031000.0,8377000.0,75.1,72.0,4.1,24.9
May-Jul 1971,40587000.0,25603000.0,24541000.0,1062000.0,14984000.0,63.1,60.5,4.1,36.9,33581000.0,25202000.0,24153000.0,1049000.0,8380000.0,75.0,71.9,4.2,25.0
